# 18 - Window size study (BGL)

How much does the window length matter on BGL? The log is cut into windows of 10, 20 (our default) and 50 lines with the same `make_windows` as in notebook 09, and PCA and Random Forest are evaluated on each with the same time-ordered split: fit on the first 60%, choose settings on the next 10%, test on the last 30%.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.bgl_features import BGL_DIR, make_windows, parse_bgl
from src.models.supervised import evaluate_rf, train_rf
from src.models.unsupervised import PCADetector, time_split_val, validated_results

DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
SIZES = [10, 20, 50]
PERCENTILES = [90, 95, 97.5, 99, 99.5]  # as in notebook 14

## Parse once

Drain only depends on the lines, not on the windows, so the log is parsed once and the same event ids are cut into windows of each size.

In [2]:
cluster_ids, alerts, timestamps, parser, line_counts = parse_bgl(BGL_DIR / "BGL.log")
print(f"{len(cluster_ids):,} lines, {parser.num_templates} templates, {sum(alerts):,} alert lines")

Parsing BGL.log: 0 lines [00:00, ? lines/s]

Parsing BGL.log: 1941423 lines [00:30, 64714.08 lines/s]

Parsing BGL.log: 1941423 lines [00:42, 64714.08 lines/s]

Parsing BGL.log: 3473709 lines [01:00, 56691.78 lines/s]

Parsing BGL.log: 3473709 lines [01:13, 56691.78 lines/s]

Parsing BGL.log: 4613185 lines [01:30, 48148.73 lines/s]

Parsing BGL.log: 4747963 lines [01:34, 50295.41 lines/s]

4,747,963 lines, 391 templates, 348,460 alert lines


## Windows

A window is anomalous if any of its lines is an alert, so longer windows are more often anomalous. The last partial window is dropped. The 20-line windows are checked against `bgl_counts.parquet` from notebook 09.

In [3]:
counts = {size: make_windows(cluster_ids, alerts, timestamps, parser.num_templates, size)[1] for size in SIZES}

saved = pd.read_parquet(PROCESSED / "bgl_counts.parquet")
assert counts[20].shape == saved.shape and (counts[20]["Label"].to_numpy() == saved["Label"].to_numpy()).all()

windows = pd.DataFrame([{"window size": size, "windows": len(c), "anomalous": c["Label"].mean(),
                         "test windows": len(time_split_val(c, order_col="window")[2])} for size, c in counts.items()])
windows.style.format({"windows": "{:,}", "test windows": "{:,}", "anomalous": "{:.1%}"}).hide(axis="index")

window size,windows,anomalous,test windows
10,"474,796",8.2%,"142,439"
20,"237,398",8.5%,"71,220"
50,"94,959",9.3%,"28,488"


## PCA and Random Forest per window size

PCA: fitted on the normal windows of the first 60%, percentile chosen by F1 on the next 10% (`validated_results`, as in notebook 14). Random Forest: class-balanced (`train_rf`), the method chosen on BGL in notebook 15, trained on the first 60%. Both are tested on the last 30%.

In [4]:
rows = []
for size, c in counts.items():
    event_cols = list(c.filter(regex=r"^E\d+$").columns)
    n_windows, anomalous = len(c), c["Label"].mean()

    _, pca = validated_results(c, {"PCA": PCADetector(variance=0.95)}, PERCENTILES, order_col="window")
    pca = pca.iloc[0]
    rows.append({"window_size": size, "windows": n_windows, "anomalous_pct": anomalous, "model": "PCA",
                 "setting": f"percentile {pca['percentile']:g}", **{m: pca[m] for m in ["precision", "recall", "f1", "false_alarm_rate"]}})

    fit, _, test = time_split_val(c, order_col="window")
    rf, _ = train_rf(fit[event_cols], fit["Label"], method="balanced")
    m = evaluate_rf(rf, test[event_cols], test["Label"])
    rows.append({"window_size": size, "windows": n_windows, "anomalous_pct": anomalous, "model": "Random Forest",
                 "setting": "method balanced", **{k: m[k] for k in ["precision", "recall", "f1", "false_alarm_rate"]}})

results = pd.DataFrame(rows)
results.round(4).to_csv(DOCS / "window_size_results.csv", index=False)
results.style.format({"windows": "{:,}", "anomalous_pct": "{:.1%}"} | {c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

window_size,windows,anomalous_pct,model,setting,precision,recall,f1,false_alarm_rate
10,"474,796",8.2%,PCA,percentile 99.5,0.091,0.996,0.167,0.749
10,"474,796",8.2%,Random Forest,method balanced,0.968,0.043,0.082,0.000
20,"237,398",8.5%,PCA,percentile 99,0.095,0.980,0.173,0.755
20,"237,398",8.5%,Random Forest,method balanced,0.955,0.072,0.134,0.000
50,"94,959",9.3%,PCA,percentile 99.5,0.101,0.922,0.182,0.771
50,"94,959",9.3%,Random Forest,method balanced,0.675,0.156,0.254,0.007


In [5]:
results.pivot(index="window_size", columns="model", values="f1").round(3)

model,PCA,Random Forest
window_size,,
10,0.167,0.082
20,0.173,0.134
50,0.182,0.254


## Findings

- **The 20-line row equals the earlier BGL results** (notebooks 14 and 15: PCA F1 0.173, Random Forest F1 0.134), so the study runs on the same data as before.
- **PCA hardly changes with the window size:** F1 0.167 / 0.173 / 0.182 and about 75% false alarms at every size. Its problem on BGL is drift between the training and test periods, not the window length.
- **Random Forest gains from longer windows:** recall rises from 0.043 (10 lines) to 0.072 (20) and 0.156 (50), and F1 from 0.082 to 0.134 and 0.254. Precision drops from 0.97 to 0.68, but the false alarm rate stays under 1%.
- **The scores aren't fully comparable across sizes.** Metrics are per window, longer windows are more often anomalous (8.2% / 8.5% / 9.3%), and one 50-line alert covers what would be five 10-line windows. Longer windows also point less precisely to the lines that caused the alert, and give 5 times fewer training windows than 10-line windows.
- **We keep 20 lines** as the default, so all earlier BGL results stay comparable. 50 lines is worth considering only if Random Forest's recall matters more than precision and localisation.